# Examples

In [1]:
import numpy as np
from data import *  
from geometry import *
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings("ignore")

## Example 1: Linear Regression (Low-dimensional)

In [2]:
from Regression import linear

### Data Generating Process

In [3]:
# number of source groups = 3, with 1000 samples each
# sigma: source group 1,3: 0.5; source group 2: 2
# target sample size = 10000
# dimension p = 5
n_list = [1000, 1000, 1000]
N = 10000  # target sample size
data = DataContainerSimu_linear_reg_lowd(n_list=n_list, N=N, p=5)
data.generate_funcs_list(seed=0)
data.generate_data(seed=0)

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target


### Implementation \& Results

In [4]:
reg = linear.lowd()
reg.fit(Xlist, Ylist, X0, loss_type='reward')
reg.infer()

In [6]:
reg.summary()

Model Summary:
Fitted Weights:

group     |        1        2        3
weight_   |   0.4567   0.3451   0.1982

Fitted Coefficients:

index     |        1        2        3        4        5
coef_     |  -0.0655  -0.0433   0.0032  -0.0018   0.0997

Confidence Intervals for each coefficient:

index     |              1              2              3              4              5
CI        | (-0.1283,-0.0027) (-0.1080,0.0214) (-0.0601,0.0665) (-0.0666,0.0629) (0.0351,0.1643)



In [ ]:
# Geometry view: the convex hull of the source coefficients (cloesest point to the origin)
beta_source = reg.beta_list
beta_ch, w_ch = nearest_on_convex_hull(beta_source)
print("Estimated coefficients on convex hull:", beta_ch)
print("Weights:", w_ch)

Estimated coefficients on convex hull: [-0.06592632 -0.0432436   0.00284681 -0.00168026  0.09944676]
Weights: [0.45700851 0.34576112 0.19723037]


In [7]:
reg = linear.lowd()
reg.fit(Xlist, Ylist, X0, loss_type='squaredloss')
#reg.infer()

In [9]:
reg.summary()

Model Summary:
Fitted Weights:

group     |        1        2        3
weight_   |   0.0000   1.0000   0.0000

Fitted Coefficients:

index     |        1        2        3        4        5
coef_     |  -0.3487  -0.1735  -0.2884  -0.1579  -0.1389

Confidence Intervals not computed. Please run infer() method.


In [ ]:
# Geometry view: the sufficiently large noise group dominates
reg.beta_list[1]

array([-0.34866929, -0.17351915, -0.2884043 , -0.15793023, -0.13894055])

In [10]:
reg = linear.lowd()
reg.fit(Xlist, Ylist, X0, loss_type='regret')
#reg.infer()

In [13]:
reg.summary()

Model Summary:
Fitted Weights:

group     |        1        2        3
weight_   |   0.3184   0.4467   0.2350

Fitted Coefficients:

index     |        1        2        3        4        5
coef_     |  -0.1004  -0.0816  -0.0368  -0.0537   0.0602

Confidence Intervals not computed. Please run infer() method.


In [ ]:
# Geometry view: the center of the minimum enclosing ball of the source coefficients
beta_source = reg.beta_list
beta_cr, r_cr, w_cr = circumcenter_3vectors(beta_source)
print("Estimated coefficients on center of minimum enclosing ball:", beta_cr)
print("Weights:", w_cr)  

Estimated coefficients on center of minimum enclosing ball: [-0.09880893 -0.08368395 -0.03569643 -0.05683792  0.06023717]
Weights: [0.3107334  0.44558458 0.24368202]


## Example 2:Linear Regression (High-dimensionl)

### Data Generating Process

In [14]:
# two source groups, each with 100 samples, and 100 target samples
n_list = [100, 100]
N = 100

data = DataContainerSimu_linear_reg_highd(n_list=n_list, N=N, p=100)
data.generate_funcs_list(seed=0)
data.generate_data(seed=0)

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target


### Implementation \& Results

In [15]:
reg = linear.highd(verbose=True)
reg.fit(Xlist, Ylist, [1,5,10,98], X0=X0)
reg.infer(M=200, alpha=0.05, alpha_thres=0.01)

## time cost: 6.6s

Argument 'loading_intercept' set to False because intercept is False
start fitting-----
======> Bias Correction for initial estimators....
---> Computing for loading (1/4)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (2/4)...
The projection direction is identified at xi = 0.060097 at step = 4.0
---> Computing for loading (3/4)...
The projection direction is identified at xi = 0.060097 at step = 4.0
---> Computing for loading (4/4)...
The projection direction is identified at xi = 0.060097 at step = 4.0
---> Computing for loading (1/4)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (2/4)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (3/4)...
The projection direction is identified at xi = 0.040065 at step = 5.0
---> Computing for loading (4/4)...
The projection direction is identified at xi = 0.060097 at step = 4.0
======> Bias 

In [16]:
reg.summary()

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.1244   0.8756

Fitted Plug-in Estimations:

index     |        1        5       10       98
coef_     |   0.0062   0.0635   0.1130   0.0284

Fitted Debiased Estimations:

index     |        1        5       10       98
coef_     |   0.2112  -0.0490   0.0186   0.4715

Confidence Intervals for each coefficient:

index     |              1              5             10             98
CI        | (-0.1514,0.5738) (-0.4764,0.3785) (-0.5055,0.5427) (0.1080,0.8349)



### Prediction

In [17]:
reg.predict()

array([ 0.61770168,  0.07562015, -0.1887311 ,  0.14399941, -0.67828077,
       -0.44789578, -0.13934558,  0.16843078, -0.77741912,  1.14980805,
        0.46831096,  0.57044041, -0.47853469, -0.56641513,  0.26428681,
       -0.16658751,  0.22940482, -0.40060955, -0.84421504, -0.53508475,
       -0.22317448,  0.44364499,  0.83598472,  0.22075631, -0.63076814,
        0.75317783,  0.06683571,  0.11320386,  0.8360526 ,  0.04662524,
       -0.06896763, -0.19391448,  0.35574663, -0.0605827 , -0.87665069,
       -0.20475603, -0.20517836, -0.55828759, -0.35727686, -0.19381614,
        0.06544052, -0.63788582,  0.67103104,  1.17324356,  0.01697582,
        0.01671607, -0.70811355, -0.13793288,  0.46519028,  0.26015514,
        0.27634214,  0.09444766,  0.49478095,  0.05218857, -0.94008998,
       -1.00736907,  0.42737417,  1.30424561, -1.37032484,  0.3580107 ,
        0.19039678,  0.18336648, -0.13223272,  0.65728401,  0.36413028,
       -0.36515295, -0.72914618,  0.48768611, -0.21218942, -0.06

## Example 3: Nonlinear Regression

In [18]:
from Regression import nonlinear

### Data Generating Process

In [ ]:
# number of source groups = 3, each with 10000 samples, and 100000 target samples
# dimension p = 5
# sigma: source group 1,3: 0.5; source group 2: 3.
data = DataContainerSimu_Nonlinear_reg(n=10000, N=100000)
data.generate_funcs_list(L=3, seed=0)
data.generate_data()

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target

### Implementation \& Prediction

In [20]:
drol = nonlinear(f_learner='xgb', w_learner='kliep')
drol.fit(Xlist,Ylist,X0, loss_type='reward')

## time cost: 8.7s

In [22]:
drol.weight_

array([0.22570861, 0.3424433 , 0.43184809])

In [23]:
drol.predict()

array([ 0.39331641,  0.37073379,  1.45123304, ..., -2.03730598,
       -1.54510459, -0.51603913])

In [ ]:
# Geometry view: the convex hull of the source coefficients (cloesest point to the origin)
pred_source = drol.pred_full_mat.T
pred_ch, w_ch = nearest_on_convex_hull(pred_source)
print("Predictions on convex hull:", pred_ch)
print("Weights:", w_ch)


Predictions on convex hull: [ 0.4093105   0.37405516  1.4294104  ... -2.0350971  -1.50406607
 -0.5645813 ]
Weights: [0.22913568 0.33249975 0.43836456]


In [24]:
drol = nonlinear(f_learner='xgb', w_learner='kliep')
drol.fit(Xlist,Ylist,X0, loss_type='squaredloss')

## time cost: 8.7s

In [26]:
drol.weight_

array([0., 1., 0.])

In [27]:
drol.predict()

array([-0.66390771,  0.15004398,  2.89583254, ..., -2.18369293,
       -4.2661047 ,  2.68939495])

In [ ]:
# Geometry view: the sufficiently large noise group dominates
pred_source = drol.pred_full_mat.T
pred_source[1]

array([-0.66390771,  0.15004398,  2.89583254, ..., -2.18369293,
       -4.2661047 ,  2.68939495])

In [28]:
drol = nonlinear(f_learner='xgb', w_learner='kliep')
drol.fit(Xlist,Ylist,X0, loss_type='regret')

## time cost: 8.7s

In [30]:
drol.weight_

array([0.40613178, 0.21867301, 0.37519521])

In [31]:
drol.predict()

array([ 0.65968302,  0.25470847,  1.40273592, ..., -2.05710611,
       -2.10659701, -1.81038661])

In [ ]:
# Geometry view: the center of the minimum enclosing ball of the source coefficients
pred_source = drol.pred_full_mat.T
pred_cr, r_cr, w_cr = circumcenter_3vectors(pred_source)
print("Predictions on center of minimum enclosing ball:", pred_cr)
print("Weights:", w_cr)

Predictions on center of minimum enclosing ball: [ 0.61270772  0.25019564  1.4571943  ... -2.06186239 -2.1890394
 -1.65294947]
Weights: [0.39229247 0.2465978  0.36110973]


## Example 4: DRlm - Classification

In [32]:
from Classification import linear

### Data Generating Process

In [33]:
# two source groups, each with 100 samples, and 1000 target samples
n = 100; p = 5; L = 2; N = 1000; K = 2
data = DataContainerSimu_linear_Cl(n=n, N=N, p=p, L=L, K=K)
data.generate_funcs_list(seed=123)
data.generate_data(seed=123)

Xlist = data.X_sources_list
Ylist = data.Y_sources_list
X0 = data.X_target


### Implementation \& Results

In [34]:
cc = linear(f_learner='linear', w_learner='linear')
cc.fit(Xlist,Ylist,X0)
cc.infer()

## time cost: 4.8s

In [35]:
cc.summary()

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.7985   0.2015

Fitted Coefficients:

Class 1 coefficients:
index     |        1        2        3        4        5
coef_     |   0.0582  -0.1451  -0.0623  -0.0652   0.3645

Class 2 coefficients:
index     |        1        2        3        4        5
coef_     |   0.3192  -0.3334  -0.1658   0.3339  -0.3211

Confidence Intervals for each coefficient:

Class 1 Confidence Intervals:
index     |              1              2              3              4              5
CIs       | (-1.113,1.287) (-1.996,1.440) (-2.633,1.996) (-1.540,3.026) (-1.153,2.826)

Class 2 Confidence Intervals:
index     |              1              2              3              4              5
CIs       | (-0.866,1.729) (-2.148,1.004) (-2.503,1.806) (-1.280,3.821) (-1.507,1.776)



In [36]:
cc.summary(
    index = [3,5], class_index=2
)

Model Summary:
Fitted Weights:

group     |        1        2
weight_   |   0.7985   0.2015

Fitted Coefficients:

Class 2 coefficients:
index     |        3        5
coef_     |  -0.1658  -0.3211

Confidence Intervals for each coefficient:

Class 2 Confidence Intervals:
index     |              3              5
CIs       | (-2.503,1.806) (-1.507,1.776)



### Prediction

In [37]:
cc.predict_proba()

array([[0.46621458, 0.33948019, 0.19430523],
       [0.33735549, 0.3403033 , 0.32234121],
       [0.49823435, 0.25034083, 0.25142482],
       ...,
       [0.3513536 , 0.46685882, 0.18178758],
       [0.34266945, 0.52183562, 0.13549493],
       [0.27539232, 0.28251022, 0.44209746]])

In [38]:
cc.predict()

array([0, 1, 0, 0, 2, 2, 0, 0, 0, 1, 2, 0, 2, 0, 1, 1, 0, 2, 0, 1, 1, 2,
       0, 2, 0, 1, 2, 2, 1, 1, 1, 1, 0, 1, 1, 0, 1, 2, 2, 2, 0, 0, 1, 1,
       1, 0, 2, 1, 1, 0, 2, 1, 1, 1, 1, 1, 1, 0, 2, 2, 1, 1, 2, 0, 1, 1,
       2, 2, 2, 2, 1, 0, 2, 2, 1, 2, 2, 1, 2, 2, 1, 1, 2, 2, 2, 2, 1, 2,
       2, 2, 0, 0, 2, 2, 2, 1, 2, 2, 2, 2, 1, 1, 0, 1, 1, 1, 2, 1, 1, 1,
       1, 1, 1, 2, 2, 2, 1, 0, 2, 2, 2, 2, 1, 2, 0, 2, 0, 1, 1, 0, 1, 0,
       1, 0, 2, 1, 1, 2, 2, 1, 2, 1, 1, 2, 2, 2, 1, 0, 1, 2, 1, 2, 2, 1,
       2, 1, 1, 0, 1, 1, 2, 2, 1, 1, 1, 1, 1, 2, 0, 2, 1, 1, 2, 1, 2, 1,
       1, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 0, 2, 1, 2, 0, 2, 0, 2, 2, 1, 0,
       0, 1, 2, 0, 2, 2, 2, 1, 1, 2, 2, 2, 1, 2, 0, 2, 0, 1, 0, 2, 2, 2,
       2, 1, 2, 1, 1, 0, 1, 1, 1, 2, 1, 2, 1, 2, 0, 2, 2, 0, 1, 0, 1, 2,
       2, 2, 2, 2, 1, 2, 2, 0, 0, 1, 1, 1, 0, 1, 1, 2, 1, 1, 2, 2, 2, 2,
       1, 2, 1, 2, 1, 1, 2, 1, 0, 1, 0, 2, 0, 0, 1, 2, 2, 2, 1, 0, 1, 2,
       2, 2, 2, 2, 2, 1, 1, 2, 2, 1, 2, 2, 1, 2, 2,